# Ingest Lambda

Packages `lambda/ingest/handler.py`, deploys it as the Lambda function `pf-ingest`, and tests it.
The function copies one batch from `landing/batch_NN/` to `bronze/patient_records/ingest_date=<date>/` (the bronze layer).

Run the cells top to bottom. Each cell is safe to run twice, and prints the AWS CLI command it runs (`$ aws ...`).

## 1. Setup

In [1]:
import json
import sys
import zipfile
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI and prints the command and its output (see infra/aws_helpers.py)
from aws_helpers import aws, aws_json, whoami, SETTINGS, LAKE

FUNCTION = SETTINGS["lambda_name"]            # pf-ingest
ROLE = "pf-lambda-role"
BUILD = ROOT / "build"                         # gitignored folder for the zip and test output
BUILD.mkdir(exist_ok=True)

print("identity:", whoami())
print("function:", FUNCTION)
print("bucket  :", LAKE)

identity: user/Ahmed
function: pf-ingest
bucket  : patientflow-datalake-zhmkbf


## 2. Package
A Lambda is deployed as a zip file. Ours holds only `handler.py`; `boto3` is already included in the Lambda Python runtime.

In [2]:
ZIP = BUILD / "pf-ingest.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(ROOT / "lambda" / "ingest" / "handler.py", arcname="handler.py")

print(f"{ZIP.name}: {ZIP.stat().st_size} bytes, contains {zipfile.ZipFile(ZIP).namelist()}")

pf-ingest.zip: 1884 bytes, contains ['handler.py']


## 3. Deploy
- **Role:** `pf-lambda-role` (what the function is allowed to do).
- **Handler:** `handler.lambda_handler` = file `handler.py`, function `lambda_handler`.
- **Environment variables:** the settings the code reads (bucket, first ingest date, number of batches).
- If the function exists, its code and configuration are updated; otherwise it is created.
  `wait` commands pause until AWS has finished, because a function cannot be changed while it is updating.

In [3]:
role_arn = aws_json("iam", "get-role", "--role-name", ROLE)["Role"]["Arn"]

CONFIG = {
    "--runtime": "python3.12",
    "--handler": "handler.lambda_handler",
    "--timeout": "60",
    "--memory-size": "256",
    "--environment": json.dumps({"Variables": {
        "LAKE_BUCKET": LAKE,
        "FIRST_INGEST_DATE": SETTINGS["first_ingest_date"],
        "NUMBER_OF_BATCHES": str(SETTINGS["number_of_batches"]),
    }}),
}
config_args = [arg for pair in CONFIG.items() for arg in pair]

exists = aws("lambda", "get-function", "--function-name", FUNCTION, check=False, show=False).returncode == 0

if exists:
    print(f"function {FUNCTION} already exists - updating code and configuration\n")
    aws("lambda", "update-function-code", "--function-name", FUNCTION,
        "--zip-file", f"fileb://build/{ZIP.name}", "--query", "LastUpdateStatus", "--output", "text")
    aws("lambda", "wait", "function-updated-v2", "--function-name", FUNCTION)
    aws("lambda", "update-function-configuration", "--function-name", FUNCTION, "--role", role_arn,
        *config_args, "--query", "LastUpdateStatus", "--output", "text")
else:
    aws("lambda", "create-function", "--function-name", FUNCTION, "--role", role_arn,
        "--zip-file", f"fileb://build/{ZIP.name}", *config_args,
        "--tags", "project=patient-flow", "--query", "State", "--output", "text")

aws("lambda", "wait", "function-updated-v2", "--function-name", FUNCTION)
aws("lambda", "wait", "function-active-v2", "--function-name", FUNCTION)

$ aws lambda get-function --function-name pf-ingest


function pf-ingest already exists - updating code and configuration

$ aws lambda update-function-code --function-name pf-ingest --zip-file fileb://build/pf-ingest.zip --query LastUpdateStatus --output text


InProgress
$ aws lambda wait function-updated-v2 --function-name pf-ingest


$ aws lambda update-function-configuration --function-name pf-ingest --role arn:aws:iam::<ACCOUNT_ID>:role/pf-lambda-role --runtime python3.12 --handler handler.lambda_handler --timeout 60 --memory-size 256 --environment {"Variables": {"LAKE_BUCKET": "patientflow-datalake-zhmkbf", "FIRST_INGEST_DATE": "2026-11-01", "NUMBER_OF_BATCHES": "4"}} --query LastUpdateStatus --output text


InProgress
$ aws lambda wait function-updated-v2 --function-name pf-ingest


$ aws lambda wait function-active-v2 --function-name pf-ingest


<aws exit code 0>

## 4. Check the deployed configuration

In [4]:
fn = aws_json("lambda", "get-function-configuration", "--function-name", FUNCTION)
assert fn["Runtime"] == "python3.12" and fn["Handler"] == "handler.lambda_handler"
assert fn["Timeout"] == 60 and fn["MemorySize"] == 256
assert fn["Role"] == role_arn
assert fn["Environment"]["Variables"]["LAKE_BUCKET"] == LAKE
assert fn["State"] == "Active"
print(f"OK  {FUNCTION}: {fn['Runtime']}, {fn['MemorySize']} MB, {fn['Timeout']} s timeout, state {fn['State']}")

OK  pf-ingest: python3.12, 256 MB, 60 s timeout, state Active


## 5. Invoke helper
`lambda invoke` runs the function once and saves its return value in a file.
A function that raises an error still returns HTTP status 200; the failure is reported in the `FunctionError` field,
so the helper checks that field too.

In [5]:
def invoke(payload):
    """Run the function with a JSON payload; return (function_error_or_None, result_dict)."""
    out_file = BUILD / "invoke_result.json"
    meta = json.loads(aws("lambda", "invoke", "--function-name", FUNCTION,
                          "--payload", json.dumps(payload), "--cli-binary-format", "raw-in-base64-out",
                          str(out_file.relative_to(ROOT)), check=True).stdout)
    assert meta["StatusCode"] == 200
    return meta.get("FunctionError"), json.loads(out_file.read_text())

## 6. Test: ingest batch 1

In [6]:
error, result = invoke({"batch": 1})
print("\nresult:", result)
assert error is None
assert result == {"status": "OK", "batch": 1, "ingest_date": "2026-11-01", "file_count": 37}
print("OK  batch 1 ingested: 37 files for ingest_date 2026-11-01")

$ aws lambda invoke --function-name pf-ingest --payload {"batch": 1} --cli-binary-format raw-in-base64-out build\invoke_result.json


{
    "StatusCode": 200,
    "ExecutedVersion": "$LATEST"
}

result: {'status': 'OK', 'batch': 1, 'ingest_date': '2026-11-01', 'file_count': 37}
OK  batch 1 ingested: 37 files for ingest_date 2026-11-01


### Check the bronze layer
The bronze partition must hold the same files as the landing batch (same names, same sizes), plus the batch metadata file.

In [7]:
def files_under(prefix):
    objects = aws_json("s3api", "list-objects-v2", "--bucket", LAKE, "--prefix", prefix).get("Contents", [])
    return {obj["Key"][len(prefix):]: obj["Size"] for obj in objects}


landing = files_under("landing/batch_01/")
bronze = files_under("bronze/patient_records/ingest_date=2026-11-01/")
metadata_size = bronze.pop("_batch_metadata.json")

assert bronze == landing, "bronze does not match landing"
print(f"OK  bronze/patient_records/ingest_date=2026-11-01 matches landing/batch_01: {len(bronze)} files, {sum(bronze.values()) / 1e6:.1f} MB")

OK  bronze/patient_records/ingest_date=2026-11-01 matches landing/batch_01: 37 files, 110.8 MB


In [8]:
metadata = json.loads(aws("s3", "cp", f"s3://{LAKE}/bronze/patient_records/ingest_date=2026-11-01/_batch_metadata.json", "-").stdout)
assert metadata["file_count"] == 37 and metadata["batch"] == 1
assert metadata["total_bytes"] == sum(landing.values())

$ aws s3 cp s3://patientflow-datalake-zhmkbf/bronze/patient_records/ingest_date=2026-11-01/_batch_metadata.json -


{"batch": 1, "ingest_date": "2026-11-01", "file_count": 37, "total_bytes": 110770899, "source_prefix": "landing/batch_01/", "ingested_at": "2026-10-26T14:52:45+00:00"}


## 7. Test: run the same batch again
The result and the files must be identical: no duplicates.

In [9]:
error, again = invoke({"batch": 1})
assert error is None and again == result
assert files_under("bronze/patient_records/ingest_date=2026-11-01/").keys() == {*landing, "_batch_metadata.json"}
print("OK  second run gave the same result and the same 38 objects (37 files + metadata file)")

$ aws lambda invoke --function-name pf-ingest --payload {"batch": 1} --cli-binary-format raw-in-base64-out build\invoke_result.json


{
    "StatusCode": 200,
    "ExecutedVersion": "$LATEST"
}


OK  second run gave the same result and the same 38 objects (37 files + metadata file)


## 8. Test: bad input must fail with a clear error

In [10]:
for bad_event in ({"batch": 0}, {"batch": 11}, {"batch": "1"}, {}):
    error, detail = invoke(bad_event)
    assert error == "Unhandled", f"{bad_event} should have failed"
    assert detail["errorType"] == "ValueError"
    print(f"OK  {bad_event!s:<15} -> {detail['errorType']}: {detail['errorMessage']}")

$ aws lambda invoke --function-name pf-ingest --payload {"batch": 0} --cli-binary-format raw-in-base64-out build\invoke_result.json


{
    "StatusCode": 200,
    "FunctionError": "Unhandled",
    "ExecutedVersion": "$LATEST"
}
OK  {'batch': 0}    -> ValueError: batch must be between 1 and 4, got 0
$ aws lambda invoke --function-name pf-ingest --payload {"batch": 11} --cli-binary-format raw-in-base64-out build\invoke_result.json


{
    "StatusCode": 200,
    "FunctionError": "Unhandled",
    "ExecutedVersion": "$LATEST"
}
OK  {'batch': 11}   -> ValueError: batch must be between 1 and 4, got 11
$ aws lambda invoke --function-name pf-ingest --payload {"batch": "1"} --cli-binary-format raw-in-base64-out build\invoke_result.json


{
    "StatusCode": 200,
    "FunctionError": "Unhandled",
    "ExecutedVersion": "$LATEST"
}
OK  {'batch': '1'}  -> ValueError: event must contain an integer "batch", got: {'batch': '1'}
$ aws lambda invoke --function-name pf-ingest --payload {} --cli-binary-format raw-in-base64-out build\invoke_result.json


{
    "StatusCode": 200,
    "FunctionError": "Unhandled",
    "ExecutedVersion": "$LATEST"
}
OK  {}              -> ValueError: event must contain an integer "batch", got: {}


## 9. Logs
The function writes one JSON line per step to CloudWatch Logs (log group `/aws/lambda/pf-ingest`).

In [11]:
aws("logs", "tail", f"/aws/lambda/{FUNCTION}", "--since", "15m", "--format", "short")

$ aws logs tail /aws/lambda/pf-ingest --since 15m --format short


2026-10-26T14:45:27 INIT_START Runtime Version: python:3.12.mainlinev2.v43	Runtime Version ARN: arn:aws:lambda:us-east-1::runtime:36aeb19f3326e129d9f096dea0a3c6a9ca8dce2610e62d2b2060747602d863ae
2026-10-26T14:45:28 START RequestId: 313069c4-815b-42bc-a5ab-2aefe0df429a Version: $LATEST
2026-10-26T14:45:28 [INFO]	2026-10-26T14:45:28.271Z	313069c4-815b-42bc-a5ab-2aefe0df429a	{"message": "ingest started", "batch": 2}
2026-10-26T14:45:28 [INFO]	2026-10-26T14:45:28.355Z	313069c4-815b-42bc-a5ab-2aefe0df429a	Found credentials in environment variables.
2026-10-26T14:45:35 [INFO]	2026-10-26T14:45:35.979Z	313069c4-815b-42bc-a5ab-2aefe0df429a	{"message": "ingest finished", "status": "OK", "batch": 2, "ingest_date": "2026-11-02", "file_count": 34}
2026-10-26T14:45:35 END RequestId: 313069c4-815b-42bc-a5ab-2aefe0df429a
2026-10-26T14:45:35 REPORT RequestId: 313069c4-815b-42bc-a5ab-2aefe0df429a	Duration: 7721.99 ms	Billed Duration: 8047 ms	Memory Size: 256 MB	Max Memory Used: 93 MB	Init Duration: 324.

<aws exit code 0>

## What this function is for

| | |
|---|---|
| Input | `{"batch": N}` |
| Output | the batch's files in `bronze/patient_records/ingest_date=<date>/` plus `_batch_metadata.json` |
| Used by | the Glue job reads `bronze/` (Phase 5); Step Functions calls this function as its first step (Phase 7) |

Only batch 1 is in `bronze/` so far. The other batches are ingested when the pipeline runs them.

**Next:** the Glue job that flattens the raw FHIR JSON into silver tables.